# Chapter 09 — SOLUTIONS: Dimensionality Reduction

**Instructor file.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
print('Setup complete!')

## Task 1 Solution: Scree Plot

In [ ]:
# 1a: Scale
X_scaled = StandardScaler().fit_transform(X)
print('Data scaled!')

# 1b: Full PCA + scree plot
pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = pca_full.explained_variance_ratio_.cumsum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='#3498db', linewidth=2)

for thresh, color in [(0.80, 'green'), (0.90, 'orange'), (0.95, 'red')]:
    n_comp = np.argmax(cumvar >= thresh) + 1
    ax.axhline(thresh, color=color, linestyle='--', alpha=0.6)
    ax.annotate(f'{thresh:.0%}: {n_comp} components', xy=(n_comp, thresh),
                xytext=(n_comp + 0.5, thresh - 0.05), fontsize=10, color=color)
    print(f'{thresh:.0%} variance → {n_comp} components')

ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA Scree Plot — Breast Cancer Dataset (30 features)')
plt.tight_layout()
plt.show()

## Task 2 Solution: 2D Visualization

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 7))
colors = {0: '#e74c3c', 1: '#2ecc71'}
labels = {0: 'malignant', 1: 'benign'}
for cls in [0, 1]:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               color=colors[cls], label=labels[cls], alpha=0.6, s=35, edgecolors='white')

var1, var2 = pca_2d.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({var1:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({var2:.1%} variance)', fontsize=12)
ax.set_title(f'PCA: 30-dim data → 2D ({var1+var2:.1%} variance retained)', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()
print('The two classes are well separated in 2D!')
print('PCA captures the most important variance → very informative projection.')

## Task 3 Solution: PCA as Preprocessing

In [ ]:
# 3a: Without PCA
pipe_nopca = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
score_nopca = cross_val_score(pipe_nopca, X, y, cv=5).mean()

# 3b: With PCA (95% variance)
pipe_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
score_pca = cross_val_score(pipe_pca, X, y, cv=5).mean()

# Fit to get n_components_
pipe_pca.fit(X, y)
n_comp = pipe_pca['pca'].n_components_

print(f'Without PCA:  {score_nopca:.4f} accuracy ({X.shape[1]} features)')
print(f'With PCA 95%: {score_pca:.4f} accuracy ({n_comp} components)')
print()
print(f'Reduced from {X.shape[1]} to {n_comp} features ({n_comp/X.shape[1]:.0%} of original)')
print('Accuracy is similar — PCA compressed 30 features into fewer while retaining performance!')
print('This matters for speed and generalization in larger, more complex datasets.')

## Bonus Solution: t-SNE Visualization

In [ ]:
# 1. Apply t-SNE to scaled data
X_scaled = StandardScaler().fit_transform(X)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_scaled)

# Also get PCA 2D projection for side-by-side comparison
X_pca = PCA(n_components=2).fit_transform(X_scaled)

# 2. Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {0: '#e74c3c', 1: '#2ecc71'}
labels = {0: 'malignant', 1: 'benign'}

for ax, (title, X_2d) in zip(axes, [('PCA', X_pca), ('t-SNE (perplexity=30)', X_tsne)]):
    for cls in [0, 1]:
        mask = y == cls
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   color=colors[cls], label=labels[cls], alpha=0.6, s=30, edgecolors='white')
    ax.set_title(f'{title}: Breast Cancer 2D Projection', fontsize=12)
    ax.legend(fontsize=11)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

plt.suptitle('PCA vs t-SNE: Class Separation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Observations:')
print('• t-SNE typically shows more pronounced cluster separation than PCA')
print('• PCA: distances between points are meaningful (linear projection)')
print('• t-SNE: distances are NOT meaningful — only neighborhood structure is!')
print('  Use t-SNE for visualization ONLY, never for downstream ML tasks.')
print('  A point near another in t-SNE space may be far in the original space.')